<a href="https://colab.research.google.com/github/vigneshshiv28/Vision-Transformer/blob/main/Vision_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torch import nn
import math
from dataclasses import dataclass

In [2]:
class PatchEmbedding(nn.Module):
  def __init__(self,patch_size,image_dim,d_model):
    super().__init__()
    assert H % patch_size == 0 and W % patch_size == 0
    self.patch_size = patch_size
    self.d_model = d_model
    self.image_dim = image_dim

    H,W,C = self.image_dim
    self.embedding = nn.Linear(self.patch_size*self.patch_size*C,self.d_model)

  def forward(self,image):

    B, H, W, C = image.shape

    assert (H, W, C) == self.image_dim


    B,H,W,C = image.shape

    n_patches_h = H // self.patch_size
    n_patches_w = W // self.patch_size

    N = (n_patches_h * n_patches_w)

    patches = image.reshape(
        B,
        n_patches_h,self.patch_size,
        n_patches_w,self.patch_size,
        C
    )

    patches = patches.permute(0, 1, 3, 2, 4, 5)

    patches = patches.reshape(B, -1, self.patch_size * self.patch_size * C)

    embedding = self.embedding(patches)

    return embedding






In [3]:
class PositionalEmbedding(nn.Module):
  def __init__(self,num_patches,d_model):
    super().__init__()

    self.d_model = d_model
    self.cls_token = nn.Parameter(torch.rand(1,1,self.input_size))
    self.positional_embedding = nn.Parameter(torch.randn(1, num_patches + 1, d_model))

  def forward(self,X):
    # X: (B, N, E)
    B = X.shape[0]

    cls_tokens = self.cls_token.expand(B, -1, -1)  # (B,1,E)

    X = torch.cat([cls_tokens, X], dim=1)  # (B, N+1, E)

    X = X + self.positional_embedding

    return X


In [4]:
class MultiheadSelfAttention(nn.Module):
  def __init__(self,n_heads,d_model,dv,dk):
    super().__init__()
    self.n_heads = n_heads
    self.d_model = d_model
    self.dk = dk
    self.dv = dv
    self.key = nn.Linear(d_model,n_heads*dk)
    self.value = nn.Linear(d_model,n_heads*dv)
    self.query = nn.Linear(d_model,n_heads*dk)

    self.out = nn.Linear(n_heads * dv, d_model)

  def forward(self,X):


     #X: (B,SL,d_model)

     B, SL, _ = X.shape


     Q = self.query(X) #Q: (B,SL,N*dk)
     K = self.key(X) #K: (B,SL,N*dk)
     V = self.value(X) #V: (B,SL,N*dv)

     Q = Q.view(B,SL,self.n_heads,self.dk) #Q: (B,SL,N,dk)
     K = K.view(B,SL,self.n_heads,self.dk) #K: (B,SL,N,dk)
     V = V.view(B,SL,self.n_heads,self.dv) #V: (B,SL,N,dk)

     Q = Q.transpose(1,2) #Q: (B,N,SL,dk)
     K = K.transpose(1,2) #K: (B,N,SL,dk)
     V = V.transpose(1,2) #V: (B,N,SL,dv)


     scores = torch.matmul(Q, K.transpose(-2,-1)) # score: (B,N,SL,SL)

     scores = scores / math.sqrt(self.dk)

     attn = torch.softmax(scores, dim=-1)#att: (B,N,SL,SL)

     out = torch.matmul(attn, V)# out:  (B, N, SL, dv)
     out = out.transpose(1,2)#out: (B,SL,N,dv)
     out = out.contiguous().view(B, SL, self.n_heads * self.dv) #out: (B,SL,N*dv)
     out = self.out(out) #out: (B,SL,d_model)


     return out


In [5]:
class AddNorm(nn.Module):
  def __init__(self,norm_shape,dropout):
    super().__init__()
    self.dropout = nn.dropout(dropout)
    self.ln = nn.LayerNorm(norm_shape)
  def forward(self,X,Y):
    return self.ln(self.dropout(Y)+X)

In [6]:
class PositionWiseFFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, X):
        return self.ffn(X)

In [7]:
class TransformerEncoder(nn.Module):
  def __init__(self,n_heads,d_model,dv,dk,dropout,d_fnn):
    super().__init__()
    self.n_heads = n_heads
    self.d_model = d_model
    self.dv = dv
    self.dk = dk
    self.dropout = dropout
    self.d_fnn = d_fnn

    self.attention = MultiheadSelfAttention(self.n_heads,self.d_model,self.dv,self.dk)
    self.norm1 = AddNorm(self.d_model,self.dropout)
    self.pffn = PositionWiseFFN(self.d_model,self.d_fnn)
    self.norm2 = AddNorm(self.d_model,self.dropout)
  def forward(self,X):
    #X: (B,SL,d_model)

    Y = self.attention(X) #Y: (B,SL,d_model)
    X1 = self.norm1(X,Y) #X1: (B,SL,d_model)
    Y1 = self.pffn(X1) #Y1: (B,SL,d_model)
    X2 = self.norm2(X1,Y1) # X2(B,SL,d_model)

    return X2


In [8]:
@dataclass
class VisionTransformerConfig:
    num_classes: int

    patch_size            = 16
    image_dim             = (224,224,3)
    d_model: int          = 768
    n_heads: int          = 12
    dk: int               = 64
    dv: int               = 64
    d_ff: int             = 2048
    dropout: float        = 0.1
    n_encoder_layers: int = 12



In [9]:
class VisionTransformer(nn.Module):
  def __init__(self,cfg: VisionTransformerConfig):
    super().__init__()
    self.patch_embedding = PatchEmbedding(cfg.patch_size, cfg.image_dim, cfg.d_model)
    self.num_patches = (cfg.image_dim[0] // cfg.patch_size) * (cfg.image_dim[1] // cfg.patch_size)
    self.positional_embedding = PositionalEmbedding(self.num_patches,cfg.d_model)
    self.encoder_layers = nn.ModuleList(
        [TransformerEncoder(cfg.n_heads,cfg.d_model,cfg.dv,cfg.dk,cfg.dropout,cfg.d_ff) for _ in range(cfg.n_encoder_layers)]
    )

    self.out = nn.Linear(cfg.d_model,cfg.num_classes)

  def forward(self,X):
    #X: (B,H,W,C)

    X = self.patch_embedding(X) #X: (B,N,d_model)
    X = self.positional_embedding(X) #X: (B,N+1,d_model)
    for layer in self.encoder_layers: # (B, N+1, d_model)
      X = layer(X)
    cls_token = X[:, 0, :] #cls_token: (B,d_model)
    out = self.out(cls_token)#out(B,n)

    return out


